### RAG Pipelines - Data Ingestion to vector DB pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory, add some meta data and storing it in a list of documents and returning it. We can use this list of documents to create vector store and then use it for question answering.
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    # print(pdf_dir,pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(pdf_files)
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # print(documents)
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            # print(all_documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

[WindowsPath('../data/pdf/oops 1 notes.pdf'), WindowsPath('../data/pdf/oops 2 notes.pdf')]
Found 2 PDF files to process

Processing: oops 1 notes.pdf
  ✓ Loaded 9 pages

Processing: oops 2 notes.pdf
  ✓ Loaded 20 pages

Total documents loaded: 29


In [15]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m89', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\oops 1 notes.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'oops 1 notes.pdf', 'file_type': 'pdf'}, page_content="Object-Oriented Programming (OOPS-1)\n\xa0\n\xa0\nIntroduction to OOPS\n\xa0\nObject-oriented programming System(OOPs)\u200b\n is a programming paradigm based\n\xa0\non the concept of \u200b\n“objects”\u200b\n and \u200b\n“classes”\u200b\n that contain data and methods. The\n\xa0\nprimary purpose of OOP is to increase the flexibility and maintainability of\n\xa0\nprograms. It is used to structure a software program into simple, reusable pieces of\n\xa0\ncode \u200b\nblueprints\u200b\n (called \u200b\nclasses\u200b\n) which are used to create individual instances of\n\xa0\nobjects\u200b\n.\n\xa0\n\xa0\nWhat is an Object?\n\xa0\nThe object is an entity that has a state and a behavior associated with it. It may be\n\xa0\nany real-world object

In [17]:
### Text Splitting get into chunks of 1000 characters with an overlap of 200 characters. This is useful for processing large documents and creating vector stores for question answering.
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,length_function=len,separators=["\n\n", "\n", " ", ""])
    split_doc=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_doc)} chunks")
    
    # show example of the chunk
    if(split_doc):
      print("\nExample chunk:")
      print(split_doc[0].page_content[:500])  # Print the first 500 characters of the first chunk
      print(f"metadata: {split_doc[0].metadata}")
    
    return split_doc

In [21]:
chunks = split_documents(all_pdf_documents)
print("Chunks",chunks)

split 29 documents into 49 chunks

Example chunk:
Object-Oriented Programming (OOPS-1)
 
 
Introduction to OOPS
 
Object-oriented programming System(OOPs)​
 is a programming paradigm based
 
on the concept of ​
“objects”​
 and ​
“classes”​
 that contain data and methods. The
 
primary purpose of OOP is to increase the flexibility and maintainability of
 
programs. It is used to structure a software program into simple, reusable pieces of
 
code ​
blueprints​
 (called ​
classes​
) which are used to create individual instances of
 
objects​
.
 
 
metadata: {'producer': 'Skia/PDF m89', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\oops 1 notes.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'oops 1 notes.pdf', 'file_type': 'pdf'}
Chunks [Document(metadata={'producer': 'Skia/PDF m89', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\oops 1 notes.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'oops 1 notes.pdf',

### Embedding and vectorStoreDB

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()